In [1]:
import pandas as pd
import numpy as np

fraud_df = pd.read_csv("../data/processed/fraud_base.csv")

fraud_df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,balance_diff_orig,balance_diff_dest,amount_balance_ratio,is_balance_drained,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,9839.64,0.0,0.057834,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,1864.28,0.0,0.087731,0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,181.00,0.0,0.994505,1,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,181.00,-21182.0,0.994505,1,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,11668.14,0.0,0.280788,0,0


In [2]:
print("Shape:", fraud_df.shape)

print("\nFraud value counts:")
print(fraud_df["isFraud"].value_counts())

print("\nFraud percentage:")
print(fraud_df["isFraud"].value_counts(normalize=True) * 100)

print("\nTransaction types:")
print(fraud_df["type"].value_counts())

Shape: (6362620, 12)

Fraud value counts:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud percentage:
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [3]:
X = fraud_df.drop("isFraud", axis=1)
y = fraud_df["isFraud"]

In [4]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:", len(numeric_cols))
print(list(numeric_cols))

print("Categorical columns:", len(categorical_cols))
print(list(categorical_cols))

Numerical columns: 10
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig', 'balance_diff_dest', 'amount_balance_ratio', 'is_balance_drained']
Categorical columns: 1
['type']


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (5090096, 11)
X_test: (1272524, 11)


In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

log_fraud_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_fraud_model.fit(X_train, y_train)

log_pred = log_fraud_model.predict(X_test)
log_proba = log_fraud_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, log_proba))
print("PR-AUC:", average_precision_score(y_test, log_proba))
print(classification_report(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))

ROC-AUC: 0.9960567809712928
PR-AUC: 0.6223817403950973
              precision    recall  f1-score   support

           0       1.00      0.97      0.98   1270881
           1       0.04      0.98      0.07      1643

    accuracy                           0.97   1272524
   macro avg       0.52      0.97      0.53   1272524
weighted avg       1.00      0.97      0.98   1272524

[[1230431   40450]
 [     39    1604]]


In [8]:
fraud_sample = fraud_df.sample(n=300000, random_state=42)

X_sample = fraud_sample.drop("isFraud", axis=1)
y_sample = fraud_sample["isFraud"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    random_state=42,
    stratify=y_sample
)

numeric_cols_s = X_sample.select_dtypes(include=["int64", "float64"]).columns
categorical_cols_s = X_sample.select_dtypes(include=["object"]).columns

In [9]:
from sklearn.ensemble import RandomForestClassifier

preprocessor_s = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols_s),
        ("cat", categorical_transformer, categorical_cols_s)
    ]
)

rf_fraud_model = Pipeline(steps=[
    ("preprocessor", preprocessor_s),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_fraud_model.fit(X_train_s, y_train_s)

rf_pred = rf_fraud_model.predict(X_test_s)
rf_proba = rf_fraud_model.predict_proba(X_test_s)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test_s, rf_proba))
print("PR-AUC:", average_precision_score(y_test_s, rf_proba))
print(classification_report(y_test_s, rf_pred))
print(confusion_matrix(y_test_s, rf_pred))

ROC-AUC: 0.9999834186151398
PR-AUC: 0.9863674796143623
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     59922
           1       0.93      0.97      0.95        78

    accuracy                           1.00     60000
   macro avg       0.96      0.99      0.97     60000
weighted avg       1.00      1.00      1.00     60000

[[59916     6]
 [    2    76]]


In [10]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(log_fraud_model, "../models/fraud_detection_model.pkl")

print("Fraud detection model saved successfully!")

Fraud detection model saved successfully!


In [11]:
import joblib

loaded_fraud_model = joblib.load("../models/fraud_detection_model.pkl")

sample = X_test.iloc[[0]]

prediction = loaded_fraud_model.predict(sample)[0]
probability = loaded_fraud_model.predict_proba(sample)[:, 1][0]

print("Prediction:", prediction)
print("Fraud Probability:", probability)

if probability >= 0.8:
    risk_level = "Critical Risk"
elif probability >= 0.5:
    risk_level = "High Risk"
elif probability >= 0.3:
    risk_level = "Medium Risk"
else:
    risk_level = "Low Risk"

print("Risk Level:", risk_level)

Prediction: 0
Fraud Probability: 1.3832665275540153e-05
Risk Level: Low Risk
